# Anomaly detection with TabFORGE

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SilenceX12138/TabFORGE/blob/master/docs/tutorial/notebooks/anomaly_detection/01_anomaly_detection.ipynb)

> **Colab setup:** Open this notebook with the badge, then follow the [tutorial setup guide](https://silencex12138.github.io/TabFORGE/#/tutorials.md) to install `tabforge-ml[tutorials]` from PyPI in this runtime. Select a GPU for pretrained workloads.

## 1. Task definition

Anomaly detection learns normality structure without target labels. `TabFORGEAnomalyDetector` fits an
unsupervised TabFORGE embedding workflow and an Isolation Forest head. It follows sklearn's `+1` inlier and
`-1` anomaly convention.

In [ ]:
%pip install "tabforge-ml[tutorials]"

In [ ]:
import os
import tempfile
from pathlib import Path

import numpy as np
from tabforge import (
    TabFORGEAnomalyDetector,
    TabFORGEArchitectureConfig,
    TabFORGEEmbeddingConfig,
    TabFORGERuntimeConfig,
    TabFORGETrainingConfig,
)

RANDOM_STATE = 7
MAX_TRAIN_ROWS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_ROWS", "256"))
MAX_STEPS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_STEPS", "10"))
DEVICE = os.environ.get("TABFORGE_TUTORIAL_DEVICE", "auto")
MODEL_PATH = os.environ.get("TABFORGE_TUTORIAL_MODEL_PATH", "mock")
CHECKPOINT = None if MODEL_PATH == "mock" else "pretrained"

## 2. Load and split with TabCamel

This compact example treats the minority German Credit class as the anomaly class. Labels define the
evaluation target and training contamination; they are never supplied to the detector.

In [ ]:
from tabcamel.data.dataset import TabularDataset

dataset = TabularDataset("credit-g", task_type="classification")
split = dataset.split("stratified", train_size=0.8, random_state=RANDOM_STATE)
train_set, test_set = split["train_set"], split["test_set"]
X_train = train_set.X_df.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
y_train = train_set.y_s.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
X_test = test_set.X_df.reset_index(drop=True)
y_test = test_set.y_s.reset_index(drop=True)
categorical_features = tuple(dataset.categorical_feature_list)
minority_class = y_train.value_counts().idxmin()
train_anomaly = (y_train == minority_class).astype(int)
test_anomaly = (y_test == minority_class).astype(int)
contamination = float(train_anomaly.mean())
print({"train": X_train.shape, "test": X_test.shape, "contamination": contamination})

## 3. Minimal API and fitting

`contamination="auto"` is the label-free default. Here the training prevalence sets a reproducible
threshold for F1 evaluation. Continuous scores remain suitable for AUROC and AUPRC.

In [ ]:
detector = TabFORGEAnomalyDetector(
    contamination=contamination,
    categorical_features=categorical_features,
    embedding_config=TabFORGEEmbeddingConfig(model_path=MODEL_PATH, n_folds=0 if MODEL_PATH == "mock" else 2),
    architecture_config=(
        TabFORGEArchitectureConfig(
            embedding_dimension=8 if MODEL_PATH == "mock" else None,
            decoder_layers=1,
            decoder_heads=2,
            decoder_ffn_factor=2,
            denoiser_layers=1,
            denoiser_heads=2,
            denoiser_ffn_factor=2,
        )
        if MODEL_PATH == "mock"
        else None
    ),
    training_config=TabFORGETrainingConfig(max_steps=MAX_STEPS, batch_size=128),
    runtime_config=TabFORGERuntimeConfig(device=DEVICE),
    random_state=RANDOM_STATE,
).fit(X_train, checkpoint=CHECKPOINT)
normality = detector.score_samples(X_test)
decision = detector.decision_function(X_test)
labels = detector.predict(X_test)

## 4. Evaluation

Negating `score_samples` yields an anomaly score where larger means more anomalous. AUPRC accompanies
AUROC because anomaly data is imbalanced. The F1 threshold comes from training contamination.

In [ ]:
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score

anomaly_score = -normality
predicted_anomaly = (labels == -1).astype(int)
print(
    {
        "auroc": roc_auc_score(test_anomaly, anomaly_score),
        "auprc": average_precision_score(test_anomaly, anomaly_score),
        "f1": f1_score(test_anomaly, predicted_anomaly),
        "decision_shape": decision.shape,
    }
)

## 5. sklearn interoperability and checkpoints

The detector supports cloning, parameter inspection, `fit_predict`, all standard outlier-scoring methods,
and exact checkpoint restoration of both TabFORGE and the Isolation Forest head.

In [ ]:
from sklearn.base import clone

assert np.isclose(clone(detector).get_params()["contamination"], contamination)
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = detector.save_checkpoint(Path(directory))
    restored = TabFORGEAnomalyDetector.restore_from_checkpoint(checkpoint_path, device=DEVICE)
    restored_scores = restored.score_samples(X_test.iloc[:8])
print({"checkpoint": checkpoint_path.name, "restored_scores": restored_scores.shape})

## 6. Reproducibility notes

Keep the split, minority-class definition, checkpoint revision, embedding choice, contamination, and
random seed fixed across methods. Repeat several seeds and report AUROC and AUPRC with mean and standard
deviation; select any F1 threshold from training or validation data only.